# The Two-Site Hubbard Model

### Section 1.0 - Import Libraries

In [1]:
# ============================================================
# Import model.py and utils.py
# ============================================================

from model import * 
from utils import *

In [ ]:
# ============================================================
# Import libraries
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import EstimatorV2 as Estimator
import matplotlib.pyplot as plt
from qiskit.quantum_info import Statevector

from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_nature.second_q.operators import ElectronicIntegrals, FermionicOp ,SparseLabelOp, PolynomialTensor, tensor_ordering
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature import LineLattice

from pyscf.scf.hf import SCF

In [3]:
# ============================================================
# Connect to IBM instance
# ============================================================

your_api_key = "wqtZu4aZo_rgCXU43dzxFuLf2_cGeBkkbnih6VmGAHLM"
your_crn = "crn:v1:bluemix:public:quantum-computing:us-east:a/d4f95db0515b47b7ba61dba8a424f873:944debad-b5dc-41ac-af39-90aee831d32b::"

QiskitRuntimeService.save_account(
     channel="ibm_quantum_platform",
     token=your_api_key,
     instance=your_crn,
     overwrite=True
     )

### Section 2.0 - Define Hubbard Model Problem

In [7]:
def get_Hubbard_model(n_sites: int, site_potential: float, coupling_energy: float, periodic_bool: bool):
    """
    A function to define the Hubbard model geometry and energies

    Args:
    n_sites = the number of sites in the model.
    site_potential = the potential energy associated with each site.
    coupling_energy = the interaction energy between electrons on different sites.
    periodic_bool = parameter to say whether periodic boundary conditions are enforced or not.

    Outputs:
    The definition of the Hubbard model of interest.
    """

    model = Hubbard1D(nsites=n_sites,
                          nelectrons=(1, 1),
                          U=site_potential,
                          t=coupling_energy,
                          periodic=periodic_bool)

    return model

### Section 3 - Construct Hamiltonian

In [6]:
def get_Hamiltonian(Hubbard_model):
    """
    A function to return the qubitised Hamiltonian of a given Hubbard model.

    Args:
    Hubbard_model = the defintiion of the Hubbard model.

    Outputs:
    The qubitised Hamiltonian corresponding to the Hubbard model of interest.
    """

    one_e_int, two_e_int = Hubbard_model.one_two_electron_integrals() # Calculate the one-electron and two-electron integrals
    
    # Create an object to encode the one-electron and two-electron integrals in Physicists' notation
    integrals = ElectronicIntegrals.from_raw_integrals(h1_a = one_e_int,
                                                       h1_b = one_e_int,
                                                       h2_aa = two_e_int,
                                                       h2_bb = two_e_int,
                                                       h2_ba = two_e_int, auto_index_order=True)

    H_fermi = ElectronicIntegrals(integrals).second_q_op()

    qubit_mapper = JordanWignerMapper()

    H_qubit = qubit_mapper.map(H_fermi)

    return H_qubit